# lightgbm for AM-I, AM-II

In [1]:
import os
import random
import json
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import optuna
import lightgbm as lgb
from concurrent.futures import ProcessPoolExecutor, as_completed
from sklearn.model_selection import KFold, learning_curve
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.impute import SimpleImputer
import logging
import warnings
from datetime import datetime
import hashlib
import pickle
import sys

# 抑制警告
warnings.filterwarnings('ignore')

# -------------------- Global Fixed Random Seed --------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# 为optuna设置种子
optuna.logging.set_verbosity(optuna.logging.WARNING)

# -------------------- 版本和配置信息 --------------------
VERSION = "1.0.0"
EXPERIMENT_CONFIG = {
    "version": VERSION,
    "seed": SEED,
    "numeric_features": ['MolWt', 'logP', 'TPSA', 'H_bond_donors', 'H_bond_acceptors'],
    "morgan_fp_features": [f'fp_{i}' for i in range(1024)],
    "other_fp_features": [f'col{i}' for i in range(823)],
    "target_column": 'UV_RT-s',
    "optuna_params": {
        "n_trials": 30,
        "cv_folds": 5,
        "direction": "maximize"
    },
    "total_expected_models": 151  # 150个临时模型 + 1个最终模型
}

# -------------------- Path Preparation --------------------
DATA_DIR = "./1-train_test_split"
OUTPUT_FOLDER = './2-lgb-models'
MODEL_DIR = OUTPUT_FOLDER
os.makedirs(MODEL_DIR, exist_ok=True)

# 创建配置文件夹
CONFIG_DIR = os.path.join(MODEL_DIR, "experiment_configs")
os.makedirs(CONFIG_DIR, exist_ok=True)

# -------------------- Feature Column Definitions --------------------
NUMERIC_FEATS = EXPERIMENT_CONFIG["numeric_features"]
MORGAN_FP = EXPERIMENT_CONFIG["morgan_fp_features"]
OTHER_FP = EXPERIMENT_CONFIG["other_fp_features"]
FEATURE_COLS = NUMERIC_FEATS + OTHER_FP + MORGAN_FP
TARGET_COL = EXPERIMENT_CONFIG["target_column"]

# Logging
logging.basicConfig(
    filename="./2-lgb-models/lgb_train.log",
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    filemode="a"
)

# ---------- 重现性辅助函数 ----------
def set_all_seeds(seed=42):
    """设置所有随机种子以确保重现性"""
    random.seed(seed)
    np.random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    
def generate_experiment_id(tag):
    """为实验生成唯一ID"""
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    return f"{tag}_{timestamp}_{SEED}"

def save_experiment_config(config, filepath):
    """保存实验配置"""
    with open(filepath, 'w') as f:
        json.dump(config, f, indent=2, default=str)

def compute_data_hash(X_train, X_test, y_train, y_test):
    """计算数据的哈希值以确保数据一致性（处理混合类型数据）"""
    try:
        # 对于numpy数组，确保转换为字节
        def array_to_bytes(arr):
            if hasattr(arr, 'tobytes'):
                return arr.tobytes()
            elif isinstance(arr, np.ndarray):
                # 处理非数值类型
                return arr.astype(str).tobytes()
            elif isinstance(arr, (list, tuple)):
                return str(arr).encode('utf-8')
            else:
                return str(arr).encode('utf-8')
        
        # 安全地转换所有数据
        data_bytes = b''
        for data in [X_train, X_test, y_train, y_test]:
            if isinstance(data, np.ndarray):
                # 检查数组数据类型
                if data.dtype.kind in ['O', 'U', 'S']:  # 对象、Unicode、字节字符串
                    # 对字符串类型进行编码
                    flat_data = data.flatten()
                    for item in flat_data:
                        if isinstance(item, str):
                            data_bytes += item.encode('utf-8')
                        elif isinstance(item, bytes):
                            data_bytes += item
                        else:
                            data_bytes += str(item).encode('utf-8')
                else:
                    # 数值类型
                    data_bytes += data.tobytes()
            elif isinstance(data, (pd.DataFrame, pd.Series)):
                # Pandas数据框或序列
                data_bytes += data.values.tobytes()
            else:
                # 其他类型
                data_bytes += str(data).encode('utf-8')
        
        return hashlib.md5(data_bytes).hexdigest()[:16]
    except Exception as e:
        logging.warning(f"Failed to compute data hash: {str(e)}")
        return "hash_calculation_failed"

# ---------- Plotting ----------
IPHONE_COLORS = {'scatter': '#007AFF', 'line': '#AEAEB2', 'text': '#000000'}

def iphone_style_ax(ax):
    """应用iPhone风格的坐标轴美学"""
    ax.tick_params(axis='both', direction='out', length=6, width=2, labelsize=16)
    for spine in ['top', 'right', 'bottom', 'left']:
        ax.spines[spine].set_visible(True)
        ax.spines[spine].set_linewidth(2)  # 设置边框线宽
    ax.grid(False)

def plot_scatter_and_residuals(y_true, y_pred, save_folder, base_name):
    """绘制散点图和残差图，完全遵循参考代码的样式规范"""
    try:
        # 检查数据有效性
        if len(y_true) == 0 or len(y_pred) == 0:
            logging.warning(f"[{base_name}] No data for plotting")
            return
        
        # 转换为numpy数组并检查NaN/Inf
        y_true = np.array(y_true).astype(float)
        y_pred = np.array(y_pred).astype(float)
        
        # 移除NaN和Inf
        valid_mask = np.isfinite(y_true) & np.isfinite(y_pred)
        if np.sum(valid_mask) < 2:  # 至少需要2个点绘图
            logging.warning(f"[{base_name}] Not enough valid data for plotting")
            return
            
        y_true_valid = y_true[valid_mask]
        y_pred_valid = y_pred[valid_mask]
        
        # 计算指标
        r2 = r2_score(y_true_valid, y_pred_valid)
        mae = mean_absolute_error(y_true_valid, y_pred_valid)
        
        # ============ 1. 散点图 (完全遵循参考代码样式) ============
        plt.figure(figsize=(6, 6))
        ax = plt.gca()
        
        # 应用iPhone风格坐标轴设置
        iphone_style_ax(ax)
        
        # 设置等纵横比
        ax.set_aspect('equal', adjustable='box')
        
        # 散点图绘制规范
        plt.scatter(
            y_true_valid, y_pred_valid,  # x轴: 真实值, y轴: 预测值
            alpha=0.8,                   # 透明度: 80%
            s=70,                       # 点大小: 70
            color=IPHONE_COLORS['scatter'],  # 颜色: iPhone蓝 (#007AFF)
            edgecolors='none'
        )
        
        # 理想拟合线 (对角线)
        lims = [min(y_true_valid.min(), y_pred_valid.min()), 
                max(y_true_valid.max(), y_pred_valid.max())]
        plt.plot(lims, lims,
                linestyle='--',          # 虚线样式
                color=IPHONE_COLORS['line'],  # 颜色: iPhone灰 (#AEAEB2)
                linewidth=3)            # 线宽: 3
        
        # 坐标轴标签 (粗体)
        plt.xlabel("True RT (s)", fontsize=18, fontweight='bold')
        plt.ylabel("Predicted RT (s)", fontsize=18, fontweight='bold')
        
        # 设置坐标轴范围
        margin = (lims[1] - lims[0]) * 0.05
        plt.xlim([lims[0] - margin, lims[1] + margin])
        plt.ylim([lims[0] - margin, lims[1] + margin])
        
        # 添加R²和MAE文本
        plt.text(
            0.05, 0.95,                # 位置: 左上角 (5%, 95%)
            f"R² = {r2:.3f}\nMAE = {mae:.2f}",  # 显示3位有效数字
            transform=ax.transAxes,
            va='top',
            fontsize=16,
            color=IPHONE_COLORS['text']  # 颜色: 黑色 (#000000)
        )
        
        # 保存散点图
        plt.tight_layout()
        plt.savefig(os.path.join(save_folder, f"{base_name}_scatter.png"), dpi=600)
        plt.close()
        
        # ============ 2. 残差图 (保持原样式，但使用iPhone风格) ============
        residuals = y_pred_valid - y_true_valid
        plt.figure(figsize=(6, 6))
        ax = plt.gca()
        
        # 应用iPhone风格坐标轴设置
        iphone_style_ax(ax)
        
        plt.scatter(y_pred_valid, residuals, 
                   alpha=0.8, s=70, 
                   color=IPHONE_COLORS['scatter'],
                   edgecolors='none')
        plt.axhline(0, linestyle='--', 
                   color=IPHONE_COLORS['line'], 
                   linewidth=3)
        
        plt.xlabel("Predicted RT (s)", fontsize=18, fontweight='bold')
        plt.ylabel("Residuals (s)", fontsize=18, fontweight='bold')
        
        # 计算残差范围
        residual_range = max(abs(residuals.min()), abs(residuals.max()))
        plt.ylim([-residual_range * 1.1, residual_range * 1.1])
        
        # 添加指标文本
        plt.text(0.05, 0.95, 
                f"R² = {r2:.3f}\nMAE = {mae:.2f}",
                transform=ax.transAxes, 
                va='top', 
                fontsize=16,
                color=IPHONE_COLORS['text'])
        
        plt.tight_layout()
        plt.savefig(os.path.join(save_folder, f"{base_name}_residuals.png"), dpi=600)
        plt.close()
        
        logging.info(f"[{base_name}] Plots saved successfully with iPhone style (R²={r2:.3f}, MAE={mae:.2f})")
        
    except Exception as e:
        logging.error(f"[{base_name}] Error in plotting: {str(e)}")

def plot_learning_curve(estimator, X, y, save_folder, base_name):
    """绘制学习曲线，增加健壮性检查"""
    try:
        # 检查数据集大小
        n_samples = len(X)
        if n_samples < 20:  # 数据集太小，不绘制学习曲线
            logging.warning(f"[{base_name}] Dataset too small ({n_samples} samples) for learning curve")
            return
            
        # 检查y是否有足够的变化
        if np.std(y) < 1e-6:  # 标准差太小
            logging.warning(f"[{base_name}] Target variable has no variance")
            return
            
        # 设置训练集大小（适应小型数据集）
        train_sizes = np.linspace(0.2, 1.0, 5)
        min_train_size = max(10, int(n_samples * 0.2))  # 至少10个样本
        
        # 计算学习曲线
        train_sizes, train_scores, val_scores = learning_curve(
            estimator, X, y, 
            cv=KFold(n_splits=min(5, n_samples//10), shuffle=True, random_state=SEED),
            scoring='r2', 
            n_jobs=1, 
            train_sizes=train_sizes,
            error_score='raise'  # 如果失败会抛出异常
        )
        
        # 检查是否有NaN值
        if np.any(np.isnan(train_scores)) or np.any(np.isnan(val_scores)):
            logging.warning(f"[{base_name}] NaN values in learning curve scores")
            return
            
        plt.figure(figsize=(6, 4))
        ax = plt.gca()
        iphone_style_ax(ax)
        
        plt.plot(train_sizes, train_scores.mean(1), 'o-', label='Train', 
                linewidth=2, color=IPHONE_COLORS['scatter'])
        plt.plot(train_sizes, val_scores.mean(1), 'o-', label='Cross-val', 
                linewidth=2, color=IPHONE_COLORS['line'])
        
        # 添加标准差阴影
        plt.fill_between(train_sizes, 
                        train_scores.mean(1) - train_scores.std(1),
                        train_scores.mean(1) + train_scores.std(1),
                        alpha=0.2, color=IPHONE_COLORS['scatter'])
        plt.fill_between(train_sizes,
                        val_scores.mean(1) - val_scores.std(1),
                        val_scores.mean(1) + val_scores.std(1),
                        alpha=0.2, color=IPHONE_COLORS['line'])
        
        plt.xlabel("Training Set Size", fontsize=14, fontweight='bold')
        plt.ylabel("R² Score", fontsize=14, fontweight='bold')
        plt.title(f"Learning Curve ({base_name})", fontsize=16, fontweight='bold')
        plt.legend(fontsize=12)
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        
        # 设置y轴范围
        all_scores = np.concatenate([train_scores.flatten(), val_scores.flatten()])
        valid_scores = all_scores[np.isfinite(all_scores)]
        if len(valid_scores) > 0:
            y_min, y_max = valid_scores.min(), valid_scores.max()
            y_range = y_max - y_min
            plt.ylim([y_min - 0.1*y_range, y_max + 0.1*y_range])
        
        plt.savefig(os.path.join(save_folder, f"{base_name}_learning_curve.png"), dpi=600)
        plt.close()
        logging.info(f"[{base_name}] Learning curve saved successfully")
        
    except Exception as e:
        logging.warning(f"[{base_name}] Could not plot learning curve: {str(e)}")

# ---------- Optuna ----------
class ReproducibleOptunaObjective:
    """可重现的Optuna目标函数"""
    def __init__(self, X_train, y_train, lgb_jobs, tag):
        self.X_train = X_train
        self.y_train = y_train
        self.lgb_jobs = lgb_jobs
        self.tag = tag
        self.models_created = 0
        self.fold_scores = []
        self.trial_params_history = []
        
    def __call__(self, trial):
        """Optuna目标函数，每个trial训练5个模型（5折交叉验证）"""
        params = {
            'n_estimators': trial.suggest_int('n_estimators', 100, 1500),
            'max_depth': trial.suggest_int('max_depth', 3, 15),
            'learning_rate': trial.suggest_float('learning_rate', 1e-3, 0.3, log=True),
            'subsample': trial.suggest_float('subsample', 0.6, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
            'lambda_l1': trial.suggest_float('lambda_l1', 1e-4, 10.0, log=True),
            'lambda_l2': trial.suggest_float('lambda_l2', 1e-4, 10.0, log=True),
            'min_child_samples': trial.suggest_int('min_child_samples', 5, 100),
            'random_state': SEED,
            'n_jobs': self.lgb_jobs,
            'verbosity': -1
        }
        
        # 记录参数
        self.trial_params_history.append({
            'trial_number': trial.number,
            'params': params.copy()
        })
        
        # 使用固定种子的KFold
        kf = KFold(
            n_splits=min(EXPERIMENT_CONFIG["optuna_params"]["cv_folds"], 
                        len(self.X_train)//20), 
            shuffle=True, 
            random_state=SEED
        )
        r2_list = []
        
        for fold_idx, (tr_idx, val_idx) in enumerate(kf.split(self.X_train)):
            X_tr, X_val = self.X_train[tr_idx], self.X_train[val_idx]
            y_tr, y_val = self.y_train[tr_idx], self.y_train[val_idx]
            
            # 检查验证集大小
            if len(X_val) < 5:
                continue
                
            # 为每个模型设置随机种子
            fold_seed = SEED + trial.number * 100 + fold_idx * 10
            model = lgb.LGBMRegressor(**params)
            model.set_params(**{'random_state': fold_seed})
            
            try:
                # 训练模型
                model.fit(
                    X_tr, y_tr,
                    eval_set=[(X_val, y_val)],
                    eval_metric='rmse',
                    callbacks=[
                        lgb.early_stopping(50, verbose=False),
                        lgb.log_evaluation(0)
                    ]
                )
                
                # 评估
                y_pred = model.predict(X_val)
                r2_val = r2_score(y_val, y_pred)
                
                if not np.isnan(r2_val):
                    r2_list.append(r2_val)
                    self.fold_scores.append({
                        'trial': trial.number,
                        'fold': fold_idx,
                        'r2': float(r2_val),
                        'n_train': len(X_tr),
                        'n_val': len(X_val)
                    })
                
                self.models_created += 1
                
                if trial.number < 5:  # 只记录前5个trial的详细日志
                    logging.debug(f"[{self.tag}] Trial {trial.number}, Fold {fold_idx}: R²={r2_val:.4f}")
                    
            except Exception as e:
                logging.warning(f"[{self.tag}] Trial {trial.number}, Fold {fold_idx} failed: {str(e)}")
                continue
        
        # 如果所有折叠都失败，返回一个低分
        if len(r2_list) == 0:
            return -10.0
        
        avg_r2 = np.mean(r2_list)
        logging.info(f"[{self.tag}] Trial {trial.number}: Average R²={avg_r2:.4f} from {len(r2_list)} folds")
        return avg_r2

# ---------- Single Dataset Processing ----------
TOTAL_CORES_TO_USE = 26

def process_dataset(train_file, lgb_jobs):
    """处理单个数据集，增强错误处理和恢复能力"""
    experiment_id = None
    try:
        set_all_seeds(SEED)
        
        tag = train_file.replace("_train.csv", "")
        train_path = os.path.join(DATA_DIR, train_file)
        test_path = train_path.replace("_train.csv", "_test.csv")
        base_path = os.path.join(MODEL_DIR, tag)
        
        # 生成实验ID
        experiment_id = generate_experiment_id(tag)
        logging.info(f"[{experiment_id}] Starting processing")
        print(f"[{experiment_id}] Starting processing")
        
        # 1. 检查文件存在性
        if not os.path.isfile(train_path):
            return {"tag": tag, "experiment_id": experiment_id, "status": "error", "message": f"Missing {train_path}"}
        
        if not os.path.isfile(test_path):
            return {"tag": tag, "experiment_id": experiment_id, "status": "error", "message": f"Missing {test_path}"}
        
        # 2. 读取数据
        try:
            train_df = pd.read_csv(train_path)
            test_df = pd.read_csv(test_path)
        except Exception as e:
            return {"tag": tag, "experiment_id": experiment_id, "status": "error", "message": f"Failed to read CSV: {str(e)}"}
        
        # 3. 检查目标列
        if TARGET_COL not in train_df.columns or TARGET_COL not in test_df.columns:
            return {"tag": tag, "experiment_id": experiment_id, "status": "error", "message": f"Missing {TARGET_COL} column"}
        
        # 4. 准备特征
        missing_cols = [col for col in FEATURE_COLS if col not in train_df.columns]
        if missing_cols:
            logging.warning(f"[{experiment_id}] Missing columns: {missing_cols[:5]}...")
            # 使用存在的列
            available_cols = [col for col in FEATURE_COLS if col in train_df.columns]
        else:
            available_cols = FEATURE_COLS
        
        if len(available_cols) < 10:
            return {"tag": tag, "experiment_id": experiment_id, "status": "error", "message": f"Too few features: {len(available_cols)}"}
        
        # 5. 提取特征和目标
        X_train = train_df[available_cols].values
        y_train = train_df[TARGET_COL].values
        X_test = test_df[available_cols].values
        y_test = test_df[TARGET_COL].values
        
        # 计算数据哈希
        try:
            data_hash = compute_data_hash(X_train, X_test, y_train, y_test)
        except Exception as hash_error:
            logging.warning(f"[{experiment_id}] Hash calculation failed, using placeholder: {str(hash_error)}")
            data_hash = f"hash_error_{datetime.now().timestamp()}"
        
        logging.info(f"[{experiment_id}] Data shape: train={X_train.shape}, test={X_test.shape}, data_hash={data_hash}")
        
        # 6. 数据检查
        if len(X_train) < 10:
            return {"tag": tag, "experiment_id": experiment_id, "status": "warn", "message": f"Train set too small: {len(X_train)} samples"}
        
        if np.std(y_train) < 1e-6:
            return {"tag": tag, "experiment_id": experiment_id, "status": "warn", "message": "Target has no variance"}
        
        # 7. 数据清洗和填充
        imputer = SimpleImputer(strategy="median")
        X_train = imputer.fit_transform(X_train)
        X_test = imputer.transform(X_test)
        
        # 检查NaN
        if np.any(np.isnan(X_train)) or np.any(np.isnan(y_train)):
            logging.warning(f"[{experiment_id}] NaN values found, using SimpleImputer")
            # 使用更健壮的策略
            X_train = np.nan_to_num(X_train, nan=np.nanmedian(X_train))
            X_test = np.nan_to_num(X_test, nan=np.nanmedian(X_test))
            y_train = np.nan_to_num(y_train, nan=np.nanmedian(y_train))
        
        # 8. 创建并运行Optuna优化
        logging.info(f"[{experiment_id}] Starting Optuna optimization")
        
        # 创建目标函数实例
        objective_func = ReproducibleOptunaObjective(X_train, y_train, lgb_jobs, tag)
        
        try:
            study = optuna.create_study(
                direction="maximize",
                sampler=optuna.samplers.TPESampler(seed=SEED),
                study_name=f"{experiment_id}_study",
                load_if_exists=False  # 确保每次都重新开始
            )
            
            # 运行30次trial，每次trial训练5个模型（5折交叉验证）
            study.optimize(
                objective_func,
                n_trials=EXPERIMENT_CONFIG["optuna_params"]["n_trials"],
                show_progress_bar=False,
                callbacks=[save_optuna_callback]  # 保存回调
            )
            
            best_params = study.best_params.copy()
            best_params.update({
                'random_state': SEED, 
                'n_jobs': lgb_jobs,
                'verbosity': -1
            })
            
            logging.info(f"[{experiment_id}] Optuna optimization completed")
            logging.info(f"[{experiment_id}] Models created during optimization: {objective_func.models_created}")
            logging.info(f"[{experiment_id}] Best params: {best_params}")
            
            # 保存优化历史
            optuna_history = {
                'experiment_id': experiment_id,
                'best_params': best_params,
                'best_value': float(study.best_value),
                'trial_params_history': objective_func.trial_params_history,
                'fold_scores': objective_func.fold_scores,
                'total_models_created': objective_func.models_created,
                'data_hash': data_hash,
                'seed': SEED
            }
            
            # 保存Optuna历史
            optuna_history_path = os.path.join(CONFIG_DIR, f"{experiment_id}_optuna_history.pkl")
            with open(optuna_history_path, 'wb') as f:
                pickle.dump(optuna_history, f)
                
        except Exception as e:
            logging.warning(f"[{experiment_id}] Optuna failed, using default params: {str(e)}")
            # 使用默认参数
            best_params = {
                'n_estimators': 500,
                'max_depth': 7,
                'learning_rate': 0.05,
                'subsample': 0.8,
                'colsample_bytree': 0.8,
                'lambda_l1': 0.01,
                'lambda_l2': 0.01,
                'min_child_samples': 20,
                'random_state': SEED,
                'n_jobs': lgb_jobs,
                'verbosity': -1
            }
        
        # 9. 最终训练 - 用全部训练数据训练1个最终模型
        logging.info(f"[{experiment_id}] Starting final training with best parameters")
        try:
            final_model = lgb.LGBMRegressor(**best_params)
            final_model.fit(
                X_train, y_train,
                eval_metric='rmse',
                callbacks=[lgb.log_evaluation(0)]
            )
            
            # 记录最终模型信息
            final_model_info = {
                'experiment_id': experiment_id,
                'model_type': 'LightGBM',
                'parameters': best_params,
                'training_samples': len(X_train),
                'features_used': len(available_cols),
                'data_hash': data_hash,
                'seed': SEED,
                'total_temp_models': objective_func.models_created if 'objective_func' in locals() else 0,
                'final_model_id': f"{experiment_id}_final"
            }
            
        except Exception as e:
            return {"tag": tag, "experiment_id": experiment_id, "status": "error", "message": f"Model training failed: {str(e)}"}
        
        # 10. 预测和评估
        y_pred = final_model.predict(X_test)
        
        # 检查预测结果
        if np.any(np.isnan(y_pred)) or np.any(np.isinf(y_pred)):
            logging.warning(f"[{experiment_id}] NaN/Inf in predictions, clipping")
            y_pred = np.clip(y_pred, np.nanpercentile(y_pred, 1), np.nanpercentile(y_pred, 99))
            y_pred = np.nan_to_num(y_pred, nan=np.nanmedian(y_pred))
        
        rmse = np.sqrt(mean_squared_error(y_test, y_pred))
        r2 = r2_score(y_test, y_pred)
        mae = mean_absolute_error(y_test, y_pred)
        
        logging.info(f"[{experiment_id}] Final model metrics - RMSE={rmse:.4f}, R2={r2:.4f}, MAE={mae:.4f}")
        
        # 11. 保存模型和结果
        # 创建模型目录
        model_dir = os.path.dirname(base_path)
        os.makedirs(model_dir, exist_ok=True)
        
        # 保存模型
        txt_path = f"{base_path}_lgb.txt"
        final_model.booster_.save_model(txt_path)
        joblib.dump(final_model, f"{base_path}_model.joblib")
        
        # 保存特征列表和imputer
        joblib.dump(available_cols, f"{base_path}_feature_list.pkl")
        joblib.dump(imputer, f"{base_path}_imputer.pkl")
        
        # 保存实验配置
        experiment_config = EXPERIMENT_CONFIG.copy()
        experiment_config.update({
            'experiment_id': experiment_id,
            'dataset': tag,
            'data_hash': data_hash,
            'final_model_info': final_model_info,
            'best_params': best_params,
            'metrics': {
                'rmse': float(rmse),
                'r2': float(r2),
                'mae': float(mae)
            }
        })
        
        config_path = os.path.join(CONFIG_DIR, f"{experiment_id}_config.json")
        save_experiment_config(experiment_config, config_path)
        
        # 保存预测结果
        test_df['pred_UV_RT-s'] = y_pred
        test_df.to_csv(f"{base_path}_test_predictions.csv", index=False)
        
        # 12. 绘图 - 使用完全遵循参考代码样式的散点图
        plot_scatter_and_residuals(y_test, y_pred, MODEL_DIR, tag)
        
        # 只有在训练集足够大时才绘制学习曲线
        if len(X_train) >= 50:
            plot_learning_curve(final_model, X_train, y_train, MODEL_DIR, tag)
        
        # 13. 保存指标
        metrics = {
            "experiment_id": experiment_id,
            "rmse": float(rmse),
            "r2": float(r2),
            "mae": float(mae),
            "n_train": int(len(X_train)),
            "n_test": int(len(X_test)),
            "n_features": int(len(available_cols)),
            "data_hash": data_hash,
            "temp_models_count": objective_func.models_created if 'objective_func' in locals() else 0,
            "final_model_trained": True,
            "seed": SEED
        }
        
        with open(f"{base_path}_metrics.json", "w") as f:
            json.dump(metrics, f, indent=2)
        
        logging.info(f"[{experiment_id}] Processing completed successfully")
        logging.info(f"[{experiment_id}] Total temporary models during optimization: {objective_func.models_created if 'objective_func' in locals() else 0}")
        
        return {
            "tag": tag, 
            "experiment_id": experiment_id, 
            "status": "success", 
            **metrics
        }
        
    except Exception as e:
        error_msg = f"Unexpected error: {str(e)}"
        logging.error(f"[{experiment_id if experiment_id else tag}] {error_msg}", exc_info=True)
        return {
            "tag": tag, 
            "experiment_id": experiment_id if experiment_id else "unknown",
            "status": "error", 
            "message": error_msg
        }

def save_optuna_callback(study, trial):
    """保存Optuna研究状态的回调函数"""
    try:
        # 每5个trial保存一次
        if trial.number % 5 == 0:
            checkpoint_path = os.path.join(CONFIG_DIR, f"optuna_checkpoint_{study.study_name}_{trial.number}.pkl")
            with open(checkpoint_path, 'wb') as f:
                pickle.dump(study, f)
    except Exception as e:
        logging.warning(f"Failed to save Optuna checkpoint: {str(e)}")

# ---------- Main ----------
def main():
    """主函数，增加整体错误处理"""
    try:
        # 保存全局配置
        global_config_path = os.path.join(CONFIG_DIR, "global_experiment_config.json")
        save_experiment_config(EXPERIMENT_CONFIG, global_config_path)
        
        # 记录开始时间
        start_time = datetime.now()
        logging.info(f"=== Starting experiment at {start_time} ===")
        logging.info(f"Version: {VERSION}")
        logging.info(f"Seed: {SEED}")
        logging.info(f"Expected total models per dataset: {EXPERIMENT_CONFIG['total_expected_models']}")
        print(f"\n=== Starting LGBM Training Experiment ===")
        print(f"Version: {VERSION}")
        print(f"Random Seed: {SEED}")
        print(f"Expected models per dataset: {EXPERIMENT_CONFIG['total_expected_models']}")
        print(f"Start time: {start_time}")
        
        train_files = [f for f in os.listdir(DATA_DIR) if f.endswith("_train.csv")]
        
        if not train_files:
            logging.error("No train files found in data directory")
            print("Error: No train files found in data directory")
            return
        
        logging.info(f"Found {len(train_files)} train files")
        print(f"Found {len(train_files)} train files")
        
        # 配置并行处理
        max_workers = min(2, len(train_files))  # 最多2个worker
        lgb_jobs = max(1, TOTAL_CORES_TO_USE // max_workers)
        
        logging.info(f"Config: workers={max_workers}, lgb_jobs={lgb_jobs}")
        print(f"Config: workers={max_workers}, lgb_jobs={lgb_jobs}")
        
        results = []
        with ProcessPoolExecutor(max_workers=max_workers) as exe:
            futures = {exe.submit(process_dataset, f, lgb_jobs): f for f in train_files}
            
            for i, fut in enumerate(as_completed(futures), 1):
                try:
                    result = fut.result(timeout=3600)  # 1小时超时
                    results.append(result)
                    
                    status = result.get('status', 'unknown')
                    tag = result.get('tag', 'unknown')
                    exp_id = result.get('experiment_id', 'unknown')
                    
                    if status == 'success':
                        r2 = result.get('r2', 0)
                        rmse = result.get('rmse', 0)
                        temp_models = result.get('temp_models_count', 0)
                        print(f"[{i}/{len(train_files)}] ✓ {tag} ({exp_id}): R²={r2:.4f}, RMSE={rmse:.4f}, TempModels={temp_models}")
                    elif status == 'warn':
                        msg = result.get('message', 'No message')
                        print(f"[{i}/{len(train_files)}] ⚠ {tag} ({exp_id}): {msg}")
                    else:
                        msg = result.get('message', 'No message')
                        print(f"[{i}/{len(train_files)}] ✗ {tag} ({exp_id}): {msg}")
                        
                except Exception as e:
                    logging.error(f"Error processing future: {str(e)}")
                    results.append({
                        "tag": futures.get(fut, "unknown"),
                        "experiment_id": "unknown",
                        "status": "error",
                        "message": f"Future processing failed: {str(e)}"
                    })
        
        # 保存结果
        results_path = os.path.join(MODEL_DIR, "training_results.json")
        with open(results_path, "w") as f:
            json.dump(results, f, indent=2)
        
        # 统计结果
        success = sum(1 for r in results if r.get('status') == 'success')
        warn = sum(1 for r in results if r.get('status') == 'warn')
        error = sum(1 for r in results if r.get('status') == 'error')
        
        # 计算平均指标
        success_results = [r for r in results if r.get('status') == 'success']
        if success_results:
            avg_r2 = np.mean([r.get('r2', 0) for r in success_results])
            avg_rmse = np.mean([r.get('rmse', 0) for r in success_results])
            avg_temp_models = np.mean([r.get('temp_models_count', 0) for r in success_results])
            logging.info(f"Average R2: {avg_r2:.4f}, Average RMSE: {avg_rmse:.4f}, Average temp models: {avg_temp_models:.1f}")
        
        end_time = datetime.now()
        duration = end_time - start_time
        
        logging.info(f"=== Experiment completed at {end_time} ===")
        logging.info(f"Duration: {duration}")
        logging.info(f"Final stats: success={success}, warn={warn}, error={error}")
        
        print(f"\n=== Training Summary ===")
        print(f"Experiment completed at: {end_time}")
        print(f"Duration: {duration}")
        print(f"Success: {success}/{len(train_files)}")
        print(f"Warnings: {warn}")
        print(f"Errors: {error}")
        
        if success_results:
            print(f"\nAverage Metrics:")
            print(f"  R²: {avg_r2:.4f}")
            print(f"  RMSE: {avg_rmse:.4f}")
            print(f"  Temporary models per dataset: {avg_temp_models:.1f}")
            print(f"  Total temporary models: {sum([r.get('temp_models_count', 0) for r in success_results])}")
            print(f"  Final models: {success}")
        
        print(f"\nConfiguration and logs saved in: {CONFIG_DIR}")
        print(f"Results saved in: {results_path}")
        
    except Exception as e:
        logging.error(f"Main function failed: {str(e)}", exc_info=True)
        print(f"Error in main function: {str(e)}")
        sys.exit(1)

if __name__ == "__main__":
    main()

/home/xuxianyan/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



=== Starting LGBM Training Experiment ===
Version: 1.0.0
Random Seed: 42
Expected models per dataset: 151
Start time: 2026-02-11 19:33:32.927443
Found 2 train files
Config: workers=2, lgb_jobs=13
[AM-I-filtered_with_labels_k4_20260211_193332_42] Starting processing[AM-II-filtered_with_labels_k4_20260211_193332_42] Starting processing

[1/2] ✓ AM-II-filtered_with_labels_k4 (AM-II-filtered_with_labels_k4_20260211_193332_42): R²=0.8719, RMSE=3.1855, TempModels=150
[2/2] ✓ AM-I-filtered_with_labels_k4 (AM-I-filtered_with_labels_k4_20260211_193332_42): R²=0.8908, RMSE=4.4503, TempModels=150

=== Training Summary ===
Experiment completed at: 2026-02-11 19:36:28.889939
Duration: 0:02:55.962496
Success: 2/2
Warnings: 0
Errors: 0

Average Metrics:
  R²: 0.8813
  RMSE: 3.8179
  Temporary models per dataset: 150.0
  Total temporary models: 300
  Final models: 2

Configuration and logs saved in: ./2-lgb-models/experiment_configs
Results saved in: ./2-lgb-models/training_results.json


# lgb for AM-III, AM-IV, AM-V, AM-VI

In [1]:
import os
import random
import json
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import optuna
import lightgbm as lgb
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.impute import SimpleImputer
import logging
from scipy import stats


# -------------------- Global Fixed Random Seeds --------------------
SEED = 42

def set_all_seeds(seed=42):
    """设置所有可能的随机种子"""
    random.seed(seed)
    np.random.seed(seed)
    # LightGBM 特定种子
    os.environ['PYTHONHASHSEED'] = str(seed)
    os.environ['LIGHTGBM_SEED'] = str(seed)
    os.environ['LIGHTGBM_DETERMINISTIC'] = 'true'

# 初始化设置
set_all_seeds(SEED)

# -------------------- Paths and Datasets --------------------
OUTPUT_FOLDER = "./2-lgb-models-other4"
MODEL_DIR = OUTPUT_FOLDER
os.makedirs(MODEL_DIR, exist_ok=True)

# Fixed four dataset names (files should be .csv with same names)
DATASETS = [
    "AM-III-filtered",
    "AM-IV-filtered",
    "AM-V-filtered", 
    "AM-VI-filtered"
]

# -------------------- Logging Configuration --------------------
logging.basicConfig(
    filename="./2-lgb-models-other4/lgb_other4_train.log",
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    filemode="a"
)

# -------------------- Feature Column Definitions --------------------
NUMERIC_FEATS = ['MolWt', 'logP', 'TPSA', 'H_bond_donors', 'H_bond_acceptors']
MORGAN_FP = [f'fp_{i}' for i in range(1024)]
OTHER_FP = [f'col{i}' for i in range(823)]
FEATURE_COLS = NUMERIC_FEATS + OTHER_FP + MORGAN_FP
TARGET_COL = 'UV_RT-s'

# -------------------- iPhone Color Scheme (Updated for consistency) --------------------
IPHONE_COLORS = {
    "scatter": "#007AFF",  # iPhone blue
    "line": "#AEAEB2",     # iPhone gray
    "text": "#000000",     # Black
    "residual": "#34C759"  # iPhone green
}

# -------------------- Get Base Name from Dataset Name --------------------
def get_base_name(dataset_name):
    """
    从数据集名称获取基础文件名（移除.csv扩展名）
    例如: "AM-I-filtered_with_labels_k4_test.csv" -> "AM-I-filtered_with_labels_k4_test"
          "AM-IV-filtered" -> "AM-IV-filtered"
    """
    if dataset_name.endswith('.csv'):
        return dataset_name[:-4]
    return dataset_name

# -------------------- Enhanced Scatter Plot Function --------------------
def plot_scatter_with_statistics(y_true, y_pred, outer_metrics, save_path, dataset_name):
    """
    绘制散点图并显示详细的统计信息
    使用Outer CV的平均性能指标，而不是OOF总体指标
    
    参数:
        y_true: 真实值数组
        y_pred: OOF预测值数组（用于散点图）
        outer_metrics: 包含Outer CV统计信息的字典
        save_path: 保存路径
        dataset_name: 数据集名称
    """
    # 创建图形，指定尺寸
    fig, (ax_scatter, ax_residual) = plt.subplots(1, 2, figsize=(12, 6))
    
    # ========== 使用Outer CV的平均性能指标 ==========
    r2_mean = outer_metrics["summary"]["r2"]["mean"]
    r2_std = outer_metrics["summary"]["r2"]["std"]
    mae_mean = outer_metrics["summary"]["mae"]["mean"]
    mae_std = outer_metrics["summary"]["mae"]["std"]
    rmse_mean = outer_metrics["summary"]["rmse"]["mean"]
    rmse_std = outer_metrics["summary"]["rmse"]["std"]
    pearson_mean = outer_metrics["summary"]["pearson_corr"]["mean"]
    pearson_std = outer_metrics["summary"]["pearson_corr"]["std"]
    
    # 计算OOF的MAPE（用于残差图）
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    
    # ========== 散点图 (左) ==========
    ax_scatter.tick_params(axis='both', direction='out', length=6, width=2, labelsize=12)
    for spine in ['top', 'right', 'bottom', 'left']:
        ax_scatter.spines[spine].set_visible(True)
        ax_scatter.spines[spine].set_linewidth(2)
    
    ax_scatter.grid(False)
    
    # 散点图
    scatter = ax_scatter.scatter(
        y_true, y_pred,
        alpha=0.8,
        s=60,
        color=IPHONE_COLORS['scatter'],
        edgecolors='none'
    )
    
    # 理想拟合线
    xymin = min(y_true.min(), y_pred.min())
    xymax = max(y_true.max(), y_pred.max())
    data_range = xymax - xymin
    pad = data_range * 0.05 if data_range > 0 else 1
    lim_min = xymin - pad
    lim_max = xymax + pad
    
    ax_scatter.plot(
        [lim_min, lim_max], [lim_min, lim_max],
        linestyle='--',
        color=IPHONE_COLORS['line'],
        linewidth=2
    )
    
    # 轴标签
    ax_scatter.set_xlabel("True RT (s)", fontsize=14, fontweight='bold')
    ax_scatter.set_ylabel("Predicted RT (s)", fontsize=14, fontweight='bold')
    ax_scatter.set_xlim([lim_min, lim_max])
    ax_scatter.set_ylim([lim_min, lim_max])
    
    # 统计信息文本 - 使用Outer CV平均值
    stats_text = (f"R² = {r2_mean:.4f} ± {r2_std:.4f}\n"
                  f"MAE = {mae_mean:.2f} ± {mae_std:.2f} s\n"
                  f"RMSE = {rmse_mean:.2f} ± {rmse_std:.2f} s\n"
                  f"Pearson r = {pearson_mean:.4f} ± {pearson_std:.4f}\n"
                  f"n = {len(y_true)}")
    ax_scatter.text(
        0.05, 0.95,
        stats_text,
        transform=ax_scatter.transAxes,
        verticalalignment='top',
        fontsize=11,  # 稍微缩小字体以适应更多文本
        color=IPHONE_COLORS['text'],
        bbox=dict(facecolor='white', alpha=0.8, edgecolor='none', pad=5)
    )
    
    # ========== 残差图 (右) ==========
    residuals = y_pred - y_true
    
    ax_residual.tick_params(axis='both', direction='out', length=6, width=2, labelsize=12)
    for spine in ['top', 'right', 'bottom', 'left']:
        ax_residual.spines[spine].set_visible(True)
        ax_residual.spines[spine].set_linewidth(2)
    
    ax_residual.grid(False)
    
    # 残差散点图
    ax_residual.scatter(
        y_pred, residuals,
        alpha=0.6,
        s=50,
        color=IPHONE_COLORS['scatter'],
        edgecolors='none'
    )
    
    # 零线
    ax_residual.axhline(y=0, linestyle='--', color=IPHONE_COLORS['line'], linewidth=2)
    
    # 残差统计
    residual_mean = np.mean(residuals)
    residual_std = np.std(residuals)
    
    ax_residual.set_xlabel("Predicted RT (s)", fontsize=14, fontweight='bold')
    ax_residual.set_ylabel("Residuals (Predicted - True)", fontsize=14, fontweight='bold')
    
    # 残差统计文本 - 使用OOF统计
    residual_stats = f"Mean = {residual_mean:.2f} s\nStd = {residual_std:.2f} s\nMAPE = {mape:.2f}%"
    ax_residual.text(
        0.05, 0.95,
        residual_stats,
        transform=ax_residual.transAxes,
        verticalalignment='top',
        fontsize=12,
        color=IPHONE_COLORS['text'],
        bbox=dict(facecolor='white', alpha=0.8, edgecolor='none', pad=5)
    )
    
    # 设置残差图的y轴范围
    residual_range = max(abs(residuals.min()), abs(residuals.max()))
    ax_residual.set_ylim([-residual_range*1.1, residual_range*1.1])
    
    # 添加标题 - 明确标注是Outer CV性能
    fig.suptitle(f"Outer 5-Fold CV Performance: {dataset_name}", fontsize=16, fontweight='bold', y=0.98)
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=600, bbox_inches='tight', facecolor='white')
    plt.close()
    
    logging.info(f"✅ 综合散点图已保存: {save_path}")
    logging.info(f"📊 Outer CV 性能 - R²={r2_mean:.4f}±{r2_std:.4f}, "
                 f"MAE={mae_mean:.2f}±{mae_std:.2f}, "
                 f"RMSE={rmse_mean:.2f}±{rmse_std:.2f}")
    
    return {
        'r2_mean': r2_mean,
        'r2_std': r2_std,
        'mae_mean': mae_mean,
        'mae_std': mae_std,
        'rmse_mean': rmse_mean,
        'rmse_std': rmse_std,
        'pearson_mean': pearson_mean,
        'pearson_std': pearson_std,
        'n_samples': len(y_true)
    }


# -------------------- Simple Scatter Plot (for comparison) --------------------
def plot_simple_scatter(y_true, y_pred, outer_metrics, save_path, dataset_name):
    """
    绘制简单的散点图 - 仅显示主要指标
    使用Outer CV的平均性能指标
    """
    plt.figure(figsize=(6, 6))
    ax = plt.gca()
    
    ax.tick_params(axis='both', direction='out', length=6, width=2, labelsize=16)
    for spine in ['top', 'right', 'bottom', 'left']:
        ax.spines[spine].set_visible(True)
        ax.spines[spine].set_linewidth(2)
    
    plt.grid(False)
    
    plt.scatter(
        y_true, y_pred,
        alpha=0.8,
        s=70,
        color=IPHONE_COLORS['scatter'],
        edgecolors='none'
    )
    
    xymin = min(y_true.min(), y_pred.min())
    xymax = max(y_true.max(), y_pred.max())
    data_range = xymax - xymin
    pad = data_range * 0.05 if data_range > 0 else 1
    lim_min = xymin - pad
    lim_max = xymax + pad
    
    plt.plot(
        [lim_min, lim_max], [lim_min, lim_max],
        linestyle='--',
        color=IPHONE_COLORS['line'],
        linewidth=3
    )
    
    # 使用Outer CV的平均性能指标
    r2_mean = outer_metrics["summary"]["r2"]["mean"]
    r2_std = outer_metrics["summary"]["r2"]["std"]
    mae_mean = outer_metrics["summary"]["mae"]["mean"]
    mae_std = outer_metrics["summary"]["mae"]["std"]
    rmse_mean = outer_metrics["summary"]["rmse"]["mean"]
    rmse_std = outer_metrics["summary"]["rmse"]["std"]
    
    plt.xlabel("True RT (s)", fontsize=18, fontweight='bold')
    plt.ylabel("Predicted RT (s)", fontsize=18, fontweight='bold')
    
    # 更新统计信息文本
    stats_text = (f"R² = {r2_mean:.3f} ± {r2_std:.3f}\n"
                  f"MAE = {mae_mean:.2f} ± {mae_std:.2f}") # s\n#
                  #f"RMSE = {rmse_mean:.2f} ± {rmse_std:.2f} s\n"
                  #f"n = {len(y_true)}")
    
    plt.text(
        0.05, 0.95,
        stats_text,
        transform=ax.transAxes,
        verticalalignment='top',
        fontsize=14,
        color=IPHONE_COLORS['text'],
        bbox=dict(facecolor='white', alpha=0.8, edgecolor='none', pad=5)
    )
    
    plt.xlim([lim_min, lim_max])
    plt.ylim([lim_min, lim_max])
    
    # ========== 添加等纵横比设置 ==========
    ax.set_aspect('equal', adjustable='box')
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=600, bbox_inches='tight', facecolor='white')
    plt.close()
    
    return r2_mean, r2_std, mae_mean, mae_std, rmse_mean, rmse_std


# -------------------- 简化的确定性采样器 --------------------
def create_deterministic_sampler(seed=SEED):
    """
    创建确定性的采样器，避免访问内部_rng属性
    """
    try:
        # 尝试使用标准TPESampler
        return optuna.samplers.TPESampler(seed=seed)
    except Exception as e:
        # 如果失败，使用RandomSampler作为备选
        logging.warning(f"使用TPESampler失败，使用RandomSampler: {e}")
        return optuna.samplers.RandomSampler(seed=seed)


# -------------------- Inner Objective Function --------------------
def inner_objective(trial, X_tr, y_tr, fold_seed_offset=0):
    """
    Objective function for Optuna: Perform 3-fold CV on (X_tr, y_tr), return mean RMSE (lower is better)
    """
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 200, 1500),
        'max_depth': trial.suggest_int('max_depth', 3, 15),
        'learning_rate': trial.suggest_float('learning_rate', 1e-3, 0.3, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'lambda_l1': trial.suggest_float('lambda_l1', 1e-4, 10.0, log=True),
        'lambda_l2': trial.suggest_float('lambda_l2', 1e-4, 10.0, log=True),
        'random_state': SEED + fold_seed_offset,
        'n_jobs': 1,
        'deterministic': True,
        'force_col_wise': True,
        'verbose': -1
    }
    
    kf_inner = KFold(n_splits=3, shuffle=True, random_state=SEED + fold_seed_offset)
    rmse_scores = []
    
    for inner_fold_idx, (tr_idx, va_idx) in enumerate(kf_inner.split(X_tr)):
        inner_fold_seed = SEED + fold_seed_offset * 10 + inner_fold_idx * 100
        set_all_seeds(inner_fold_seed)
        
        X_tr_i, X_va_i = X_tr[tr_idx], X_tr[va_idx]
        y_tr_i, y_va_i = y_tr[tr_idx], y_tr[va_idx]

        model = lgb.LGBMRegressor(**params)
        model.fit(
            X_tr_i, y_tr_i,
            eval_set=[(X_va_i, y_va_i)],
            eval_metric='rmse',
            callbacks=[
                lgb.early_stopping(stopping_rounds=100, verbose=False),
                lgb.log_evaluation(0)
            ]
        )
        preds = model.predict(X_va_i)
        rmse_scores.append(np.sqrt(mean_squared_error(y_va_i, preds)))
    
    return float(np.mean(rmse_scores))


# -------------------- Nested Cross-Validation (Outer 5-fold, Inner 3-fold Tuning) --------------------
def nested_cv_lightgbm(X, y, n_trials=30):
    """
    Nested CV with detailed statistics
    Returns:
      - outer_metrics: dict containing RMSE/R2/MAE per fold and mean±std
      - best_params_per_fold: list of best parameters for each outer fold
      - oof_pred: outer OOF predictions
      - fold_predictions: detailed fold information
    """
    set_all_seeds(SEED)
    
    outer_kf = KFold(n_splits=5, shuffle=True, random_state=SEED)
    metrics_per_fold = []
    best_params_per_fold = []
    oof_pred = np.zeros_like(y, dtype=float)
    fold_predictions = []
    
    for fold, (tr_idx, va_idx) in enumerate(outer_kf.split(X), start=1):
        fold_seed = SEED + fold * 1000
        set_all_seeds(fold_seed)
        
        X_tr_raw, X_va_raw = X[tr_idx], X[va_idx]
        y_tr, y_va = y[tr_idx], y[va_idx]

        imputer = SimpleImputer(strategy="median")
        X_tr = imputer.fit_transform(X_tr_raw)
        X_va = imputer.transform(X_va_raw)

        # 创建study，使用简化版采样器
        study = optuna.create_study(
            direction="minimize", 
            sampler=create_deterministic_sampler(seed=fold_seed)
        )
        
        study.optimize(
            lambda t: inner_objective(t, X_tr, y_tr, fold_seed_offset=fold), 
            n_trials=n_trials, 
            show_progress_bar=False
        )
        
        best_params = study.best_params.copy()
        best_params.update({
            'random_state': fold_seed, 
            'n_jobs': 1,
            'deterministic': True,
            'force_col_wise': True,
            'verbose': -1
        })

        set_all_seeds(fold_seed)
        model = lgb.LGBMRegressor(**best_params)
        model.fit(
            X_tr, y_tr,
            eval_metric='rmse',
            callbacks=[lgb.log_evaluation(0)]
        )
        
        pred_va = model.predict(X_va)
        oof_pred[va_idx] = pred_va

        fold_data = {
            'fold': fold,
            'fold_seed': fold_seed,
            'train_indices': tr_idx.tolist(),
            'val_indices': va_idx.tolist(),
            'y_true': y_va.tolist(),
            'y_pred': pred_va.tolist(),
            'best_params': best_params
        }
        fold_predictions.append(fold_data)
        
        rmse = float(np.sqrt(mean_squared_error(y_va, pred_va)))
        r2 = float(r2_score(y_va, pred_va))
        mae = float(mean_absolute_error(y_va, pred_va))
        mape = np.mean(np.abs((y_va - pred_va) / y_va)) * 100
        pearson_corr, _ = stats.pearsonr(y_va, pred_va)

        metrics_per_fold.append({
            "fold": fold, 
            "rmse": rmse, 
            "r2": r2, 
            "mae": mae,
            "mape": mape,
            "pearson_corr": pearson_corr,
            "n_samples": len(y_va)
        })
        best_params_per_fold.append(best_params)

        logging.info(f"[NestedCV] Fold {fold}: RMSE={rmse:.4f}, R²={r2:.4f}, MAE={mae:.4f}, Pearson r={pearson_corr:.4f}")

    # 计算详细的统计信息
    metrics_df = pd.DataFrame(metrics_per_fold)
    
    outer_summary = {
        "rmse": {
            "mean": float(metrics_df['rmse'].mean()),
            "std": float(metrics_df['rmse'].std(ddof=1)),
            "values": metrics_df['rmse'].tolist()
        },
        "r2": {
            "mean": float(metrics_df['r2'].mean()),
            "std": float(metrics_df['r2'].std(ddof=1)),
            "values": metrics_df['r2'].tolist()
        },
        "mae": {
            "mean": float(metrics_df['mae'].mean()),
            "std": float(metrics_df['mae'].std(ddof=1)),
            "values": metrics_df['mae'].tolist()
        },
        "mape": {
            "mean": float(metrics_df['mape'].mean()),
            "std": float(metrics_df['mape'].std(ddof=1)),
            "values": metrics_df['mape'].tolist()
        },
        "pearson_corr": {
            "mean": float(metrics_df['pearson_corr'].mean()),
            "std": float(metrics_df['pearson_corr'].std(ddof=1)),
            "values": metrics_df['pearson_corr'].tolist()
        }
    }

    outer_metrics = {
        "per_fold": metrics_per_fold,
        "summary": outer_summary,
        "total_seed": SEED
    }
    
    return outer_metrics, best_params_per_fold, oof_pred, fold_predictions, metrics_df


# -------------------- Select Best Configuration on Full Data --------------------
def select_best_on_full_data(X, y, n_trials=50):
    """
    Perform inner 3-fold tuning on full data
    """
    full_seed = SEED + 9999
    set_all_seeds(full_seed)
    
    imputer = SimpleImputer(strategy="median")
    X_imp = imputer.fit_transform(X)

    study = optuna.create_study(
        direction="minimize", 
        sampler=create_deterministic_sampler(seed=full_seed)
    )
    
    def full_objective(trial):
        return inner_objective(trial, X_imp, y, fold_seed_offset=9999)
    
    study.optimize(full_objective, n_trials=n_trials, show_progress_bar=False)
    
    best_params = study.best_params.copy()
    best_params.update({
        'random_state': full_seed, 
        'n_jobs': 1,
        'deterministic': True,
        'force_col_wise': True,
        'verbose': -1
    })
    
    return imputer, best_params, float(study.best_value), len(study.trials), full_seed


# -------------------- Generate Statistical Report --------------------
def generate_statistical_report(dataset_name, y_true, y_pred, outer_metrics, save_path):
    """
    生成详细的统计报告
    """
    # 计算OOF总体性能
    oof_r2 = r2_score(y_true, y_pred)
    oof_mae = mean_absolute_error(y_true, y_pred)
    oof_rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    oof_mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    oof_pearson, _ = stats.pearsonr(y_true, y_pred)
    
    # 计算残差统计
    residuals = y_pred - y_true
    residual_mean = np.mean(residuals)
    residual_std = np.std(residuals)
    residual_skew = stats.skew(residuals)
    residual_kurtosis = stats.kurtosis(residuals)
    
    # 使用Outer CV的平均性能指标
    r2_mean = outer_metrics["summary"]["r2"]["mean"]
    r2_std = outer_metrics["summary"]["r2"]["std"]
    mae_mean = outer_metrics["summary"]["mae"]["mean"]
    mae_std = outer_metrics["summary"]["mae"]["std"]
    rmse_mean = outer_metrics["summary"]["rmse"]["mean"]
    rmse_std = outer_metrics["summary"]["rmse"]["std"]
    pearson_mean = outer_metrics["summary"]["pearson_corr"]["mean"]
    pearson_std = outer_metrics["summary"]["pearson_corr"]["std"]
    
    report = {
        "dataset": dataset_name,
        "global_seed": SEED,
        "sample_statistics": {
            "n_samples": len(y_true),
            "y_true_mean": float(np.mean(y_true)),
            "y_true_std": float(np.std(y_true)),
            "y_true_min": float(np.min(y_true)),
            "y_true_max": float(np.max(y_true)),
            "y_pred_mean": float(np.mean(y_pred)),
            "y_pred_std": float(np.std(y_pred)),
            "y_pred_min": float(np.min(y_pred)),
            "y_pred_max": float(np.max(y_pred))
        },
        "outer_cv_performance": {
            "oof_overall": {
                "r2": float(oof_r2),
                "mae": float(oof_mae),
                "rmse": float(oof_rmse),
                "mape": float(oof_mape),
                "pearson_correlation": float(oof_pearson)
            },
            "fold_statistics": outer_metrics["summary"],
            "fold_consistency": {
                "rmse_cv": float(outer_metrics["summary"]["rmse"]["std"] / outer_metrics["summary"]["rmse"]["mean"] * 100),
                "r2_cv": float(outer_metrics["summary"]["r2"]["std"] / outer_metrics["summary"]["r2"]["mean"] * 100),
                "mae_cv": float(outer_metrics["summary"]["mae"]["std"] / outer_metrics["summary"]["mae"]["mean"] * 100)
            }
        },
        "residual_analysis": {
            "mean": float(residual_mean),
            "std": float(residual_std),
            "skewness": float(residual_skew),
            "kurtosis": float(residual_kurtosis),
            "shapiro_wilk_test": {
                "statistic": float(stats.shapiro(residuals).statistic),
                "p_value": float(stats.shapiro(residuals).pvalue)
            }
        },
        "performance_summary": {
            "rmse": f"{rmse_mean:.4f} ± {rmse_std:.4f}",
            "r2": f"{r2_mean:.4f} ± {r2_std:.4f}",
            "mae": f"{mae_mean:.4f} ± {mae_std:.4f}",
            "pearson_corr": f"{pearson_mean:.4f} ± {pearson_std:.4f}"
        },
        "notes": "性能指标基于Outer 5-fold CV的平均值±标准差，代表模型的真实泛化能力"
    }
    
    # 保存报告
    with open(save_path, 'w', encoding='utf-8') as f:
        json.dump(report, f, indent=2, ensure_ascii=False)
    
    logging.info(f"✅ 统计报告已保存: {save_path}")
    
    return report


# -------------------- Create Fold Performance Visualization --------------------
def create_fold_performance_visualization(metrics_df, save_path, dataset_name):
    """
    创建Fold性能对比可视化
    """
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    
    # 设置颜色
    colors = plt.cm.Set2(np.linspace(0, 1, 5))
    
    # 1. R² across folds
    axes[0, 0].bar(metrics_df['fold'], metrics_df['r2'], color=colors, edgecolor='black')
    axes[0, 0].set_xlabel('Fold', fontweight='bold', fontsize=12)
    axes[0, 0].set_ylabel('R² Score', fontweight='bold', fontsize=12)
    axes[0, 0].set_title('R² Score Across Folds', fontweight='bold', fontsize=14)
    axes[0, 0].set_ylim([0, 1])
    axes[0, 0].grid(True, alpha=0.3, linestyle='--')
    
    # 2. RMSE across folds
    axes[0, 1].bar(metrics_df['fold'], metrics_df['rmse'], color=colors, edgecolor='black')
    axes[0, 1].set_xlabel('Fold', fontweight='bold', fontsize=12)
    axes[0, 1].set_ylabel('RMSE (s)', fontweight='bold', fontsize=12)
    axes[0, 1].set_title('RMSE Across Folds', fontweight='bold', fontsize=14)
    axes[0, 1].grid(True, alpha=0.3, linestyle='--')
    
    # 3. MAE across folds
    axes[1, 0].bar(metrics_df['fold'], metrics_df['mae'], color=colors, edgecolor='black')
    axes[1, 0].set_xlabel('Fold', fontweight='bold', fontsize=12)
    axes[1, 0].set_ylabel('MAE (s)', fontweight='bold', fontsize=12)
    axes[1, 0].set_title('MAE Across Folds', fontweight='bold', fontsize=14)
    axes[1, 0].grid(True, alpha=0.3, linestyle='--')
    
    # 4. Pearson Correlation across folds
    axes[1, 1].bar(metrics_df['fold'], metrics_df['pearson_corr'], color=colors, edgecolor='black')
    axes[1, 1].set_xlabel('Fold', fontweight='bold', fontsize=12)
    axes[1, 1].set_ylabel('Pearson r', fontweight='bold', fontsize=12)
    axes[1, 1].set_title('Pearson Correlation Across Folds', fontweight='bold', fontsize=14)
    axes[1, 1].set_ylim([0, 1])
    axes[1, 1].grid(True, alpha=0.3, linestyle='--')
    
    # 添加总体标题
    fig.suptitle(f'Outer 5-Fold CV Performance: {dataset_name}', fontsize=16, fontweight='bold', y=0.98)
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=600, bbox_inches='tight', facecolor='white')
    plt.close()
    
    logging.info(f"✅ Fold性能对比图已保存: {save_path}")


# -------------------- Single Dataset Processing Pipeline --------------------
def process_dataset(dataset_name):
    base_name = get_base_name(dataset_name)
    csv_path = f"./processed_results/{dataset_name}.csv"
    base_path = os.path.join(MODEL_DIR, base_name)
    result = {"dataset": dataset_name, "base_name": base_name, "status": "ok"}

    try:
        if not os.path.isfile(csv_path):
            msg = f"Data file not found: {csv_path}"
            logging.warning(f"[{base_name}] {msg}")
            return {"dataset": dataset_name, "base_name": base_name, "status": "warn", "message": msg}

        df = pd.read_csv(csv_path)

        # Basic validation
        for col in FEATURE_COLS + [TARGET_COL]:
            if col not in df.columns:
                msg = f"Missing column: {col}"
                logging.warning(f"[{base_name}] {msg}")
                return {"dataset": dataset_name, "base_name": base_name, "status": "warn", "message": msg}

        X_all = df[FEATURE_COLS].values
        y_all = df[TARGET_COL].values

        if np.isnan(y_all).any():
            msg = "Target column contains missing values"
            logging.warning(f"[{base_name}] {msg}")
            return {"dataset": dataset_name, "base_name": base_name, "status": "warn", "message": msg}

        # ---------- Nested CV: Outer 5-fold for generalization estimate ----------
        logging.info(f"[{base_name}] Starting nested CV with seed={SEED}")
        outer_metrics, best_params_per_fold, oof_pred, fold_predictions, metrics_df = nested_cv_lightgbm(X_all, y_all, n_trials=20)

        # ---------- 生成基于Outer CV预测的真实性能散点图 ----------
        # 1. 综合散点图（带残差图）
        combined_scatter_path = os.path.join(MODEL_DIR, f"{base_name}_outer_cv_combined.png")
        combined_stats = plot_scatter_with_statistics(y_all, oof_pred, outer_metrics, combined_scatter_path, base_name)
        
        # 2. 简单散点图
        simple_scatter_path = os.path.join(MODEL_DIR, f"{base_name}_outer_cv_scatter.png")
        r2_mean, r2_std, mae_mean, mae_std, rmse_mean, rmse_std = plot_simple_scatter(
            y_all, oof_pred, outer_metrics, simple_scatter_path, base_name
        )
        
        # 3. Fold性能对比图
        fold_perf_path = os.path.join(MODEL_DIR, f"{base_name}_fold_performance.png")
        create_fold_performance_visualization(metrics_df, fold_perf_path, base_name)
        
        # ---------- 保存Fold预测结果 ----------
        fold_predictions_path = f"{base_path}_fold_predictions.json"
        with open(fold_predictions_path, 'w', encoding='utf-8') as f:
            json.dump(fold_predictions, f, indent=2, ensure_ascii=False)
        logging.info(f"[{base_name}] Fold predictions saved to: {fold_predictions_path}")

        # ---------- 生成统计报告 ----------
        report_path = f"{base_path}_statistical_report.json"
        report = generate_statistical_report(base_name, y_all, oof_pred, outer_metrics, report_path)
        
        # ---------- Final configuration selection on full data ----------
        logging.info(f"[{base_name}] Starting final model selection on full data")
        imputer_full, best_params_full, inner_best_value_full, inner_n_trials_full, full_seed = select_best_on_full_data(
            X_all, y_all, n_trials=50
        )

        # 使用最终种子训练最终模型
        set_all_seeds(full_seed)
        X_full_imp = imputer_full.fit_transform(X_all)
        final_model = lgb.LGBMRegressor(**best_params_full)
        final_model.fit(
            X_full_imp, y_all,
            eval_metric='rmse',
            callbacks=[lgb.log_evaluation(0)]
        )

        # Save model and related files
        model_path = f"{base_path}_final_model.joblib"
        feature_path = f"{base_path}_feature_list.pkl"
        imputer_path = f"{base_path}_imputer.pkl"
        oof_path = f"{base_path}_oof.npy"
        oof_csv_path = f"{base_path}_oof_predictions.csv"
        
        joblib.dump(final_model, model_path)
        joblib.dump(FEATURE_COLS, feature_path)
        joblib.dump(imputer_full, imputer_path)
        np.save(oof_path, oof_pred)
        
        # 保存CSV格式的OOF预测结果
        oof_df = pd.DataFrame({
            'index': df.index.tolist(),
            'true_rt': y_all,
            'pred_rt': oof_pred,
            'residual': oof_pred - y_all,
            'abs_residual': np.abs(oof_pred - y_all),
            'rel_error': np.abs((oof_pred - y_all) / y_all) * 100
        })
        oof_df.to_csv(oof_csv_path, index=False)
        logging.info(f"[{base_name}] OOF predictions CSV saved to: {oof_csv_path}")

        # 保存metrics - 更新使用Outer CV平均值
        metrics = {
            "dataset": dataset_name,
            "base_name": base_name,
            "target": TARGET_COL,
            "global_seed": SEED,
            "nested_cv": {
                "outer_5fold_metrics": outer_metrics,
                "best_params_per_outer_fold": best_params_per_fold,
                "outer_cv_performance_summary": {
                    "r2_mean": r2_mean,
                    "r2_std": r2_std,
                    "mae_mean": mae_mean,
                    "mae_std": mae_std,
                    "rmse_mean": rmse_mean,
                    "rmse_std": rmse_std,
                    "pearson_corr_mean": outer_metrics["summary"]["pearson_corr"]["mean"],
                    "pearson_corr_std": outer_metrics["summary"]["pearson_corr"]["std"]
                }
            },
            "final_selection_on_full_data": {
                "best_params": best_params_full,
                "inner_cv_rmse_mean": float(inner_best_value_full),
                "n_trials": inner_n_trials_full,
                "full_data_seed": full_seed
            },
            "performance_summary_formatted": report["performance_summary"],
            "notes": f"Deterministic training with SEED={SEED}. Performance metrics are based on Outer 5-fold CV mean±std."
        }
        metrics_path = f"{base_name}_metrics.json"
        metrics_full_path = os.path.join(MODEL_DIR, metrics_path)
        with open(metrics_full_path, "w", encoding="utf-8") as f:
            json.dump(metrics, f, indent=2, ensure_ascii=False)

        # 打印Outer CV统计信息 - 使用平均值
        print(f"\n📊 Outer CV Performance Summary for {base_name}:")
        print(f"   R²: {r2_mean:.4f} ± {r2_std:.4f}")
        print(f"   RMSE: {rmse_mean:.2f} ± {rmse_std:.2f} s")
        print(f"   MAE: {mae_mean:.2f} ± {mae_std:.2f} s")
        print(f"   Pearson r: {outer_metrics['summary']['pearson_corr']['mean']:.4f} ± "
              f"{outer_metrics['summary']['pearson_corr']['std']:.4f}")
        
        logging.info(f"[{base_name}] Completed. Global SEED={SEED}")
        logging.info(f"[{base_name}] Outer CV R²: {r2_mean:.4f}±{r2_std:.4f}")
        logging.info(f"[{base_name}] Outer CV RMSE: {rmse_mean:.2f}±{rmse_std:.2f}")
        logging.info(f"[{base_name}] Outer CV MAE: {mae_mean:.2f}±{mae_std:.2f}")
        
        result.update({
            "global_seed": SEED,
            "rmse_mean": outer_metrics["summary"]["rmse"]["mean"],
            "rmse_std": outer_metrics["summary"]["rmse"]["std"],
            "r2_mean": outer_metrics["summary"]["r2"]["mean"],
            "r2_std": outer_metrics["summary"]["r2"]["std"],
            "mae_mean": outer_metrics["summary"]["mae"]["mean"],
            "mae_std": outer_metrics["summary"]["mae"]["std"],
            "pearson_mean": outer_metrics["summary"]["pearson_corr"]["mean"],
            "pearson_std": outer_metrics["summary"]["pearson_corr"]["std"],
            "outer_oof_r2": report["outer_cv_performance"]["oof_overall"]["r2"],
            "outer_oof_mae": report["outer_cv_performance"]["oof_overall"]["mae"],
            "outer_oof_rmse": report["outer_cv_performance"]["oof_overall"]["rmse"],
            "performance_summary": report["performance_summary"],
            "full_data_seed": full_seed
        })
        return result

    except Exception as e:
        logging.error(f"[{base_name}] Exception: {str(e)}", exc_info=True)
        return {"dataset": dataset_name, "base_name": base_name, "status": "error", "message": str(e)}


# -------------------- Create Overall Summary Report --------------------
def create_overall_summary_report(results, output_dir):
    """
    创建所有数据集的整体汇总报告
    使用Outer CV的平均性能指标
    """
    successful_results = [r for r in results if r['status'] == 'ok']
    
    if not successful_results:
        return
    
    summary_data = []
    for r in successful_results:
        summary_data.append({
            'Dataset': r['dataset'],
            'R² (CV Mean±Std)': f"{r['r2_mean']:.4f} ± {r['r2_std']:.4f}",
            'MAE (CV Mean±Std)': f"{r['mae_mean']:.2f} ± {r['mae_std']:.2f} s",
            'RMSE (CV Mean±Std)': f"{r['rmse_mean']:.2f} ± {r['rmse_std']:.2f} s",
            'Pearson r (CV Mean±Std)': f"{r['pearson_mean']:.4f} ± {r['pearson_std']:.4f}",
            'R² (OOF)': f"{r['outer_oof_r2']:.4f}",
            'MAE (OOF)': f"{r['outer_oof_mae']:.2f} s",
            'RMSE (OOF)': f"{r['outer_oof_rmse']:.2f} s",
            'Seed': r['full_data_seed']
        })
    
    df_summary = pd.DataFrame(summary_data)
    
    # 保存为CSV
    csv_path = os.path.join(output_dir, "overall_performance_summary.csv")
    df_summary.to_csv(csv_path, index=False)
    
    # 保存为JSON
    json_path = os.path.join(output_dir, "overall_performance_summary.json")
    with open(json_path, 'w', encoding='utf-8') as f:
        json.dump(summary_data, f, indent=2, ensure_ascii=False)
    
    # 创建汇总可视化
    create_overall_comparison_plot(successful_results, output_dir)
    
    return df_summary


def create_overall_comparison_plot(results, output_dir):
    """
    创建所有数据集的性能对比图
    使用Outer CV的平均性能指标
    """
    datasets = [r['dataset'] for r in results]
    
    # 准备数据 - 使用Outer CV平均值
    r2_means = [r['r2_mean'] for r in results]
    r2_stds = [r['r2_std'] for r in results]
    rmse_means = [r['rmse_mean'] for r in results]
    rmse_stds = [r['rmse_std'] for r in results]
    
    # 创建对比图
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
    
    # R²对比
    x_pos = np.arange(len(datasets))
    ax1.bar(x_pos, r2_means, yerr=r2_stds, capsize=5, color=IPHONE_COLORS['scatter'], alpha=0.8)
    ax1.set_xlabel('Dataset', fontweight='bold', fontsize=12)
    ax1.set_ylabel('R² Score (Mean ± SD)', fontweight='bold', fontsize=12)
    ax1.set_title('Outer CV R² Score Comparison', fontweight='bold', fontsize=14)
    ax1.set_xticks(x_pos)
    ax1.set_xticklabels(datasets, rotation=45, ha='right')
    ax1.set_ylim([0, 1])
    ax1.grid(True, alpha=0.3, linestyle='--')
    
    # RMSE对比
    ax2.bar(x_pos, rmse_means, yerr=rmse_stds, capsize=5, color=IPHONE_COLORS['line'], alpha=0.8)
    ax2.set_xlabel('Dataset', fontweight='bold', fontsize=12)
    ax2.set_ylabel('RMSE (s) (Mean ± SD)', fontweight='bold', fontsize=12)
    ax2.set_title('Outer CV RMSE Comparison', fontweight='bold', fontsize=14)
    ax2.set_xticks(x_pos)
    ax2.set_xticklabels(datasets, rotation=45, ha='right')
    ax2.grid(True, alpha=0.3, linestyle='--')
    
    plt.tight_layout()
    plot_path = os.path.join(output_dir, "overall_dataset_comparison.png")
    plt.savefig(plot_path, dpi=600, bbox_inches='tight', facecolor='white')
    plt.close()
    
    logging.info(f"✅ Overall comparison plot saved to: {plot_path}")


# -------------------- Main Function --------------------
def main():
    try:
        logging.info(f"Starting serial processing of {len(DATASETS)} datasets with SEED={SEED}.")
        print(f"\n{'='*80}")
        print(f"LightGBM Nested Cross-Validation Pipeline")
        print(f"Global SEED: {SEED}")
        print(f"{'='*80}")
        print(f"Output directory: {MODEL_DIR}")
        print(f"Datasets to process: {DATASETS}")
        print(f"Target column: {TARGET_COL}")
        print(f"{'='*80}\n")
        
        results = []
        for dataset_name in DATASETS:
            print(f"\n{'='*80}")
            print(f"📁 Processing dataset: {dataset_name}")
            print(f"   Global SEED: {SEED}")
            print(f"{'='*80}")
            logging.info(f"Processing dataset: {dataset_name}")
            
            result = process_dataset(dataset_name)
            results.append(result)
            
            if result['status'] == 'ok':
                print(f"✅ Completed dataset: {dataset_name}")
                print(f"   📊 Outer CV Performance:")
                print(f"      R²: {result.get('performance_summary', {}).get('r2', 'N/A')}")
                print(f"      RMSE: {result.get('performance_summary', {}).get('rmse', 'N/A')}")
                print(f"      MAE: {result.get('performance_summary', {}).get('mae', 'N/A')}")
                print(f"      Pearson r: {result.get('performance_summary', {}).get('pearson_corr', 'N/A')}")
                print(f"   📈 OOF Performance (参考):")
                print(f"      R²: {result.get('outer_oof_r2', 0):.4f}")
                print(f"      MAE: {result.get('outer_oof_mae', 0):.2f} s")
                print(f"   🔢 Seed: {result.get('full_data_seed', 'N/A')}")
            elif result['status'] == 'warn':
                print(f"⚠️  Warning for dataset: {dataset_name}")
                print(f"   Message: {result.get('message', 'Unknown warning')}")
            else:
                print(f"❌ Error for dataset: {dataset_name}")
                print(f"   Message: {result.get('message', 'Unknown error')}")
            
            logging.info(f"Completed dataset: {dataset_name} - Status: {result['status']}")

        # Save detailed results
        summary_path = os.path.join(MODEL_DIR, "training_results.json")
        with open(summary_path, "w", encoding="utf-8") as f:
            json.dump(results, f, indent=2, ensure_ascii=False)
        
        # 创建整体汇总报告
        print(f"\n{'='*80}")
        print("📋 Overall Training Summary (Based on Outer CV Mean ± Std)")
        print(f"{'='*80}")
        
        summary_df = create_overall_summary_report(results, MODEL_DIR)
        
        if summary_df is not None:
            print("\nOuter CV Performance Summary (Mean ± Standard Deviation):")
            print("-" * 120)
            print(summary_df.to_string(index=False))
            
            # 打印详细统计
            print(f"\n{'='*80}")
            print("📈 Key Statistics (Outer CV):")
            print(f"{'='*80}")
            
            for r in results:
                if r['status'] == 'ok':
                    print(f"\nDataset: {r['dataset']}")
                    print(f"  R²: {r['r2_mean']:.4f} ± {r['r2_std']:.4f}  (OOF: {r['outer_oof_r2']:.4f})")
                    print(f"  RMSE: {r['rmse_mean']:.2f} ± {r['rmse_std']:.2f} s  (OOF: {r['outer_oof_rmse']:.2f} s)")
                    print(f"  MAE: {r['mae_mean']:.2f} ± {r['mae_std']:.2f} s  (OOF: {r['outer_oof_mae']:.2f} s)")
                    print(f"  Pearson r: {r['pearson_mean']:.4f} ± {r['pearson_std']:.4f}")
        
        ok_count = sum(1 for r in results if r.get('status') == 'ok')
        warn_count = sum(1 for r in results if r.get('status') == 'warn')
        err_count = sum(1 for r in results if r.get('status') == 'error')
        
        print(f"\n{'='*80}")
        print(f"✅ All datasets processed. Success: {ok_count}, Warnings: {warn_count}, Errors: {err_count}")
        print(f"📁 Output directory: {MODEL_DIR}")
        print(f"📊 Performance metrics are based on Outer 5-fold CV mean ± std (true generalization ability)")
        print(f"{'='*80}")
        
        logging.info(f"All datasets processed. Success: {ok_count}, Warnings: {warn_count}, Errors: {err_count}")
        
    except Exception as e:
        error_msg = f"Main function exception: {str(e)}"
        logging.error(error_msg, exc_info=True)
        print(f"❌ Error in main function: {str(e)}")


if __name__ == "__main__":
    main()

/home/xuxianyan/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



LightGBM Nested Cross-Validation Pipeline
Global SEED: 42
Output directory: ./2-lgb-models-other4
Datasets to process: ['AM-III-filtered', 'AM-IV-filtered', 'AM-V-filtered', 'AM-VI-filtered']
Target column: UV_RT-s


📁 Processing dataset: AM-III-filtered
   Global SEED: 42


[I 2026-02-28 13:50:29,131] A new study created in memory with name: no-name-e6fd9763-f6ed-45a1-9f59-5cfead5781f0
/home/xuxianyan/.local/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/xuxianyan/.local/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/xuxianyan/.local/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
[I 2026-02-28 13:50:29,446] Trial 0 finished with value: 5.082478217823606 and parameters: {'n_estimators': 616, 'max_depth': 4, 'learning_rate': 0.10779089589390585, 'subsample': 0.6646030911928217, 'colsample_bytree': 0.9800372060352323, 'lambda_l1': 0.004809281821297905, 'lambda_


📊 Outer CV Performance Summary for AM-III-filtered:
   R²: 0.8765 ± 0.0453
   RMSE: 4.45 ± 0.98 s
   MAE: 2.86 ± 0.49 s
   Pearson r: 0.9371 ± 0.0245
✅ Completed dataset: AM-III-filtered
   📊 Outer CV Performance:
      R²: 0.8765 ± 0.0453
      RMSE: 4.4501 ± 0.9786
      MAE: 2.8615 ± 0.4893
      Pearson r: 0.9371 ± 0.0245
   📈 OOF Performance (参考):
      R²: 0.8760
      MAE: 2.86 s
   🔢 Seed: 10041

📁 Processing dataset: AM-IV-filtered
   Global SEED: 42


/home/xuxianyan/.local/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/xuxianyan/.local/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/xuxianyan/.local/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
[I 2026-02-28 13:51:38,598] Trial 1 finished with value: 6.2100950949047204 and parameters: {'n_estimators': 1082, 'max_depth': 5, 'learning_rate': 0.253674287550457, 'subsample': 0.981818196128192, 'colsample_bytree': 0.7294388549076616, 'lambda_l1': 0.0017251789469707602, 'lambda_l2': 0.0009190457208788636}. Best is trial 1 with value: 6.2100950949047204.
/home/xuxianyan/.local/lib/python3.10


📊 Outer CV Performance Summary for AM-IV-filtered:
   R²: 0.7324 ± 0.0741
   RMSE: 5.23 ± 0.42 s
   MAE: 3.76 ± 0.42 s
   Pearson r: 0.8632 ± 0.0445
✅ Completed dataset: AM-IV-filtered
   📊 Outer CV Performance:
      R²: 0.7324 ± 0.0741
      RMSE: 5.2327 ± 0.4172
      MAE: 3.7604 ± 0.4194
      Pearson r: 0.8632 ± 0.0445
   📈 OOF Performance (参考):
      R²: 0.7433
      MAE: 3.76 s
   🔢 Seed: 10041

📁 Processing dataset: AM-V-filtered
   Global SEED: 42


/home/xuxianyan/.local/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/xuxianyan/.local/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/xuxianyan/.local/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
[I 2026-02-28 13:52:20,810] Trial 1 finished with value: 3.467862847670182 and parameters: {'n_estimators': 1082, 'max_depth': 5, 'learning_rate': 0.253674287550457, 'subsample': 0.981818196128192, 'colsample_bytree': 0.7294388549076616, 'lambda_l1': 0.0017251789469707602, 'lambda_l2': 0.0009190457208788636}. Best is trial 0 with value: 3.4386149541684325.
/home/xuxianyan/.local/lib/python3.10/


📊 Outer CV Performance Summary for AM-V-filtered:
   R²: 0.8312 ± 0.0299
   RMSE: 2.95 ± 0.27 s
   MAE: 2.23 ± 0.15 s
   Pearson r: 0.9157 ± 0.0125
✅ Completed dataset: AM-V-filtered
   📊 Outer CV Performance:
      R²: 0.8312 ± 0.0299
      RMSE: 2.9498 ± 0.2715
      MAE: 2.2304 ± 0.1542
      Pearson r: 0.9157 ± 0.0125
   📈 OOF Performance (参考):
      R²: 0.8336
      MAE: 2.23 s
   🔢 Seed: 10041

📁 Processing dataset: AM-VI-filtered
   Global SEED: 42


/home/xuxianyan/.local/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/xuxianyan/.local/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/xuxianyan/.local/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
[I 2026-02-28 13:52:57,312] Trial 2 finished with value: 12.254543887966639 and parameters: {'n_estimators': 416, 'max_depth': 11, 'learning_rate': 0.00404828725997507, 'subsample': 0.64942975822398, 'colsample_bytree': 0.8696785313082465, 'lambda_l1': 0.4725551664512088, 'lambda_l2': 0.012133476487665044}. Best is trial 1 with value: 7.8699580520831445.
/home/xuxianyan/.local/lib/python3.10/si


📊 Outer CV Performance Summary for AM-VI-filtered:
   R²: 0.7677 ± 0.0859
   RMSE: 8.54 ± 1.19 s
   MAE: 5.92 ± 1.22 s
   Pearson r: 0.9048 ± 0.0425
✅ Completed dataset: AM-VI-filtered
   📊 Outer CV Performance:
      R²: 0.7677 ± 0.0859
      RMSE: 8.5446 ± 1.1871
      MAE: 5.9220 ± 1.2248
      Pearson r: 0.9048 ± 0.0425
   📈 OOF Performance (参考):
      R²: 0.7823
      MAE: 5.92 s
   🔢 Seed: 10041

📋 Overall Training Summary (Based on Outer CV Mean ± Std)

Outer CV Performance Summary (Mean ± Standard Deviation):
------------------------------------------------------------------------------------------------------------------------
        Dataset R² (CV Mean±Std) MAE (CV Mean±Std) RMSE (CV Mean±Std) Pearson r (CV Mean±Std) R² (OOF) MAE (OOF) RMSE (OOF)  Seed
AM-III-filtered  0.8765 ± 0.0453     2.86 ± 0.49 s      4.45 ± 0.98 s         0.9371 ± 0.0245   0.8760    2.86 s     4.54 s 10041
 AM-IV-filtered  0.7324 ± 0.0741     3.76 ± 0.42 s      5.23 ± 0.42 s         0.8632 ± 0.0445  